In [3]:
!pip install crewai
!pip install langchain
!pip install transformers
!pip install sentence-transformers
!pip install scikit-learn

In [4]:
import json
import re
import datetime
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

Define Use Case & Agents

In [5]:
USE_CASE = "AI Research Report Generation Pipeline"

AGENTS = {
    "ResearchAgent": "Collects factual research on a topic",
    "AnalysisAgent": "Analyzes research findings",
    "ReportAgent": "Generates structured final report"
}

print("Use Case:", USE_CASE)
print("Agents:", AGENTS)

Use Case: AI Research Report Generation Pipeline
Agents: {'ResearchAgent': 'Collects factual research on a topic', 'AnalysisAgent': 'Analyzes research findings', 'ReportAgent': 'Generates structured final report'}


Create Prompt Contracts

In [6]:
prompt_contracts = {

    "ResearchAgent": {
        "purpose": "Collect factual information about a topic.",
        "allowed_actions": [
            "Search topic",
            "Summarize findings",
            "Provide references"
        ],
        "forbidden_actions": [
            "Opinionated writing",
            "Final report writing",
            "Data analysis"
        ],
        "required_inputs": ["topic"],
        "expected_outputs": ["bullet_point_research"],
        "escalation": "If insufficient data found"
    },

    "AnalysisAgent": {
        "purpose": "Analyze research findings logically.",
        "allowed_actions": [
            "Identify patterns",
            "Highlight trends",
            "Compare data"
        ],
        "forbidden_actions": [
            "New research collection",
            "Final formatting"
        ],
        "required_inputs": ["research_data"],
        "expected_outputs": ["analysis_summary"],
        "escalation": "If research data incomplete"
    },

    "ReportAgent": {
        "purpose": "Generate final structured report.",
        "allowed_actions": [
            "Format report",
            "Create headings",
            "Summarize conclusions"
        ],
        "forbidden_actions": [
            "New analysis",
            "New research"
        ],
        "required_inputs": ["analysis_summary"],
        "expected_outputs": ["final_report"],
        "escalation": "If analysis missing"
    }
}

print(json.dumps(prompt_contracts, indent=4))

{
    "ResearchAgent": {
        "purpose": "Collect factual information about a topic.",
        "allowed_actions": [
            "Search topic",
            "Summarize findings",
            "Provide references"
        ],
        "forbidden_actions": [
            "Opinionated writing",
            "Final report writing",
            "Data analysis"
        ],
        "required_inputs": [
            "topic"
        ],
        "expected_outputs": [
            "bullet_point_research"
        ],
        "escalation": "If insufficient data found"
    },
    "AnalysisAgent": {
        "purpose": "Analyze research findings logically.",
        "allowed_actions": [
            "Identify patterns",
            "Highlight trends",
            "Compare data"
        ],
        "forbidden_actions": [
            "New research collection",
            "Final formatting"
        ],
        "required_inputs": [
            "research_data"
        ],
        "expected_outputs": [
            "an

Simulated Agent Output

In [7]:
def simulate_agent(agent_name, input_data):

    if agent_name == "ResearchAgent":
        return "• AI improves automation\n• AI increases productivity\n• Ethical concerns exist"

    if agent_name == "AnalysisAgent":
        return "AI improves productivity but raises ethical concerns."

    if agent_name == "ReportAgent":
        return "Final Report:\nAI provides productivity gains while introducing ethical risks."

Scope Enforcement System

In [8]:
def validate_output(agent_name, output):

    contract = prompt_contracts[agent_name]
    violations = []

    # Check forbidden keywords
    for forbidden in contract["forbidden_actions"]:
        if forbidden.lower() in output.lower():
            violations.append(f"Forbidden action detected: {forbidden}")

    # Basic structure checks
    if agent_name == "ResearchAgent":
        if "•" not in output:
            violations.append("Expected bullet point research output")

    if agent_name == "ReportAgent":
        if "Final Report" not in output:
            violations.append("Report formatting missing")

    return violations

Run Multi-Agent Pipeline

In [9]:
topic = "Impact of AI on Business"

# Research Phase
research_output = simulate_agent("ResearchAgent", topic)
research_violations = validate_output("ResearchAgent", research_output)

# Analysis Phase
analysis_output = simulate_agent("AnalysisAgent", research_output)
analysis_violations = validate_output("AnalysisAgent", analysis_output)

# Report Phase
report_output = simulate_agent("ReportAgent", analysis_output)
report_violations = validate_output("ReportAgent", report_output)

print("Research Output:\n", research_output)
print("\nAnalysis Output:\n", analysis_output)
print("\nReport Output:\n", report_output)

print("\nViolations:")
print("Research:", research_violations)
print("Analysis:", analysis_violations)
print("Report:", report_violations)

Research Output:
 • AI improves automation
• AI increases productivity
• Ethical concerns exist

Analysis Output:
 AI improves productivity but raises ethical concerns.

Report Output:
 Final Report:
AI provides productivity gains while introducing ethical risks.

Violations:
Research: []
Analysis: []
Report: []


Prompt Drift Measurement

In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Measure Output Deviation

In [11]:
def measure_semantic_drift(expected, actual):

    expected_embedding = model.encode([expected])
    actual_embedding = model.encode([actual])

    similarity = cosine_similarity(expected_embedding, actual_embedding)[0][0]

    drift_score = 1 - similarity

    return drift_score


expected_research_template = "• Point 1\n• Point 2\n• Point 3"

drift_score = measure_semantic_drift(expected_research_template, research_output)

print("Semantic Drift Score:", drift_score)

Semantic Drift Score: 0.8923333


Drift Logging System

In [14]:
drift_log = []

def log_drift(agent, drift_score, violations):

    record = {
        "timestamp": str(datetime.datetime.now()),
        "agent": agent,
        "drift_score": float(drift_score),
        "violations": violations
    }

    drift_log.append(record)

log_drift("ResearchAgent", drift_score, research_violations)

print(json.dumps(drift_log, indent=4))

[
    {
        "timestamp": "2026-02-28 05:40:28.566236",
        "agent": "ResearchAgent",
        "drift_score": 0.8923333287239075,
        "violations": []
    }
]


Enforcement Strategy

In [15]:
DRIFT_THRESHOLD = 0.4

def enforce(agent, drift_score, violations):

    if drift_score > DRIFT_THRESHOLD:
        return "STOP: Excessive semantic drift detected."

    if len(violations) > 0:
        return "STOP: Contract violation detected."

    return "Output Approved."


decision = enforce("ResearchAgent", drift_score, research_violations)

print("Enforcement Decision:", decision)

Enforcement Decision: STOP: Excessive semantic drift detected.
